# NMF on ModulAir 00692

In [8]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [9]:
#importing data from Modulair MOD-00692
data = pd.read_csv('MOD-00692.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:12Z,577611775,2025-12-31T18:59:12Z,MOD-00692,49.9,0.2,21.729,1.378,0.226,0.085,0.028,...,34.593,22.724,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.09
2025-12-31T23:58:12Z,577611776,2025-12-31T18:58:12Z,MOD-00692,49.4,0.2,9.733,0.906,0.247,0.116,0.062,...,34.140,23.490,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.61
2025-12-31T23:57:12Z,577609805,2025-12-31T18:57:12Z,MOD-00692,49.5,0.1,7.114,0.735,0.261,0.044,0.039,...,33.655,23.823,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.58
2025-12-31T23:56:12Z,577609806,2025-12-31T18:56:12Z,MOD-00692,49.7,0.1,7.072,0.655,0.242,0.094,0.068,...,33.883,23.447,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,4.24
2025-12-31T23:55:12Z,577609807,2025-12-31T18:55:12Z,MOD-00692,50.2,0.1,7.851,0.916,0.297,0.113,0.051,...,34.810,23.385,14349.0,14350.0,14351.0,14479.0,14504.0,14554.0,14529.0,3.75


In [10]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:12Z,2025-12-31T18:59:12Z,774.794,2.896,34.593,22.724,21.729,1.378,0.226,0.085,0.028,0.034
2025-12-31T23:58:12Z,2025-12-31T18:58:12Z,739.435,2.559,34.140,23.490,9.733,0.906,0.247,0.116,0.062,0.013
2025-12-31T23:57:12Z,2025-12-31T18:57:12Z,732.687,2.768,33.655,23.823,7.114,0.735,0.261,0.044,0.039,0.032
2025-12-31T23:56:12Z,2025-12-31T18:56:12Z,739.114,2.896,33.883,23.447,7.072,0.655,0.242,0.094,0.068,0.011
2025-12-31T23:55:12Z,2025-12-31T18:55:12Z,747.379,2.907,34.810,23.385,7.851,0.916,0.297,0.113,0.051,0.016


In [11]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:12Z,774.794,2.896,34.593,22.724,21.729,1.378,0.226,0.085,0.028,0.034
1,2025-12-31T18:58:12Z,739.435,2.559,34.140,23.490,9.733,0.906,0.247,0.116,0.062,0.013
2,2025-12-31T18:57:12Z,732.687,2.768,33.655,23.823,7.114,0.735,0.261,0.044,0.039,0.032
3,2025-12-31T18:56:12Z,739.114,2.896,33.883,23.447,7.072,0.655,0.242,0.094,0.068,0.011
4,2025-12-31T18:55:12Z,747.379,2.907,34.810,23.385,7.851,0.916,0.297,0.113,0.051,0.016


In [12]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:12,774.794,2.896,34.593,22.724,21.729,1.378,0.226,0.085,0.028,0.034
1,2025-12-31 18:58:12,739.435,2.559,34.140,23.490,9.733,0.906,0.247,0.116,0.062,0.013
2,2025-12-31 18:57:12,732.687,2.768,33.655,23.823,7.114,0.735,0.261,0.044,0.039,0.032
3,2025-12-31 18:56:12,739.114,2.896,33.883,23.447,7.072,0.655,0.242,0.094,0.068,0.011
4,2025-12-31 18:55:12,747.379,2.907,34.810,23.385,7.851,0.916,0.297,0.113,0.051,0.016


In [13]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-09-06 01:00:00,688.42,10.44,34.74,1.93,10.82,0.71,0.30,0.09,0.12,0.10
1,2025-09-06 02:00:00,694.14,9.46,33.42,1.92,10.49,0.70,0.28,0.09,0.12,0.10
2,2025-09-06 03:00:00,664.56,8.88,32.78,1.93,10.08,0.68,0.28,0.09,0.12,0.10
3,2025-09-06 04:00:00,667.32,8.10,32.08,1.92,9.98,0.65,0.26,0.08,0.11,0.10
4,2025-09-06 05:00:00,667.89,7.60,29.91,1.91,10.20,0.68,0.26,0.08,0.11,0.09
...,...,...,...,...,...,...,...,...,...,...,...
2743,2025-12-31 14:00:00,679.63,32.58,28.95,1.89,8.78,0.64,0.19,0.04,0.03,0.02
2744,2025-12-31 15:00:00,690.96,32.87,28.35,1.93,7.56,0.62,0.19,0.05,0.04,0.02
2745,2025-12-31 16:00:00,705.10,33.32,27.32,2.35,8.51,0.74,0.22,0.05,0.04,0.02
2746,2025-12-31 17:00:00,762.18,34.65,23.12,2.10,9.62,0.98,0.31,0.08,0.06,0.02


In [14]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-09-06 01:00:00,688.42,10.44,34.74,1.93,10.82,0.71,0.30,0.09,0.12,0.10
2025-09-06 02:00:00,694.14,9.46,33.42,1.92,10.49,0.70,0.28,0.09,0.12,0.10
2025-09-06 03:00:00,664.56,8.88,32.78,1.93,10.08,0.68,0.28,0.09,0.12,0.10
2025-09-06 04:00:00,667.32,8.10,32.08,1.92,9.98,0.65,0.26,0.08,0.11,0.10
2025-09-06 05:00:00,667.89,7.60,29.91,1.91,10.20,0.68,0.26,0.08,0.11,0.09


In [15]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [16]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [17]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-09-06 01:00:00,0.358162,0.237219,0.514896,0.035303,0.158419,0.015322,0.016376,0.025352,0.034188,0.074074
2025-09-06 02:00:00,0.361138,0.214951,0.495331,0.035120,0.153587,0.015106,0.015284,0.025352,0.034188,0.074074
2025-09-06 03:00:00,0.345749,0.201772,0.485846,0.035303,0.147584,0.014674,0.015284,0.025352,0.034188,0.074074
2025-09-06 04:00:00,0.347185,0.184049,0.475471,0.035120,0.146120,0.014027,0.014192,0.022535,0.031339,0.074074
2025-09-06 05:00:00,0.347481,0.172688,0.443308,0.034937,0.149341,0.014674,0.014192,0.022535,0.031339,0.066667


In [18]:
data.to_csv(r'MOD-000692_timeseries_hourly_scaled.csv')

## Implementing NMF

In [19]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [20]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-09-06 01:00:00,0.362912,0.235342,0.514250,0.045239,0.152197,0.028545,0.023034,0.026276,0.028148,0.060644
2025-09-06 02:00:00,0.361803,0.214124,0.495924,0.045468,0.153416,0.025062,0.020224,0.023070,0.025751,0.059679
2025-09-06 03:00:00,0.346835,0.200851,0.486485,0.043619,0.147497,0.025132,0.020280,0.023134,0.025630,0.058222
2025-09-06 04:00:00,0.346930,0.183463,0.476483,0.043928,0.147998,0.022135,0.017862,0.020376,0.023693,0.057621
2025-09-06 05:00:00,0.345308,0.172727,0.444488,0.043981,0.152719,0.020676,0.016684,0.019033,0.022154,0.056584
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.376939,0.734363,0.420880,0.039788,0.089950,0.030195,0.024366,0.027795,0.024128,0.032997
2025-12-31 15:00:00,0.376005,0.742729,0.414207,0.039536,0.082730,0.026174,0.021121,0.024093,0.021490,0.030429
2025-12-31 16:00:00,0.385603,0.752526,0.398423,0.040792,0.094395,0.028627,0.023101,0.026352,0.022586,0.031895


In [21]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.020563,0.021802,0.152411,0.135269
1,0.017628,0.019142,0.148907,0.143331
2,0.016393,0.019195,0.146859,0.135852
3,0.014012,0.016906,0.145628,0.142110
4,0.012265,0.015792,0.136555,0.150715
...,...,...,...,...
2676,0.086772,0.023062,0.063543,0.054886
2677,0.088046,0.019991,0.060217,0.053485
2678,0.088618,0.021865,0.054487,0.063353
2679,0.091023,0.023751,0.032353,0.089784


In [22]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [23]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,2.908475,8.069763,2.640435,0.270960,0.000000,0.000000,0.000000,0.000000,0.017118,0.000000
Feature 2,0.010371,0.420349,0.000000,0.000000,1.983378,1.309302,1.056531,1.205235,0.791146,0.568045
Feature 3,0.841587,0.000000,3.017859,0.106393,0.000000,0.000000,0.000000,0.000000,0.069206,0.185093
Feature 4,1.290841,0.445306,0.000000,0.173368,0.805481,0.000000,0.000000,0.000000,0.000000,0.148217


In [24]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-09-06 01:00:00,0.020563,0.021802,0.152411,0.135269
2025-09-06 02:00:00,0.017628,0.019142,0.148907,0.143331
2025-09-06 03:00:00,0.016393,0.019195,0.146859,0.135852
2025-09-06 04:00:00,0.014012,0.016906,0.145628,0.142110
2025-09-06 05:00:00,0.012265,0.015792,0.136555,0.150715
...,...,...,...,...
2025-12-31 14:00:00,0.086772,0.023062,0.063543,0.054886
2025-12-31 15:00:00,0.088046,0.019991,0.060217,0.053485
2025-12-31 16:00:00,0.088618,0.021865,0.054487,0.063353


In [25]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,2.908475,8.069763,2.640435,0.270960,0.000000,0.000000,0.000000,0.000000,0.017118,0.000000
Factor 2,0.010371,0.420349,0.000000,0.000000,1.983378,1.309302,1.056531,1.205235,0.791146,0.568045
Factor 3,0.841587,0.000000,3.017859,0.106393,0.000000,0.000000,0.000000,0.000000,0.069206,0.185093
Factor 4,1.290841,0.445306,0.000000,0.173368,0.805481,0.000000,0.000000,0.000000,0.000000,0.148217


In [26]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.559052,0.000549,0.149307,0.289241,0.001850
no,0.476075,0.000000,0.172536,0.355092,-0.003704
no2,0.927331,0.013302,0.000000,0.059653,-0.000287
o3,0.486989,0.000000,0.513732,0.000000,-0.000721
bin0,0.000000,0.373382,0.000000,0.641905,-0.015288
bin1,0.000000,1.081335,0.000000,0.000000,-0.081335
bin2,0.000000,1.142064,0.000000,0.000000,-0.142064
bin3,0.000000,1.016578,0.000000,0.000000,-0.016578
bin4,0.056473,0.718764,0.210733,0.000000,0.014031
bin5,0.000000,0.313233,0.342088,0.345979,-0.001300


In [27]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')